# Beyond-Flesch ELECTRA Gradio Demo

Standalone Colab notebook for inference only. It loads the saved ELECTRA + ScalarMix DANN checkpoint from Google Drive and launches a Gradio UI.

Default checkpoint: `best_model.pt` from the CNN + OneStop + RACE training run,.

Inputs: sentence, paragraph, or passage. Output: `elementary`, `middle`, or `high` with probabilities.

In [ ]:
!pip install -q transformers torch gradio

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────
DRIVE_OUT_DIR = "/content/drive/MyDrive/BeyondFK/trail/electra_cnn_ose_race_train_multi_ood_dann"

# Use best_model.pt for the best validation / aggregate OOD checkpoint.
# Change to "phase2_final.pt" if you specifically want DANN-final weights.
CKPT_NAME = "best_model.pt"

MODEL_NAME = "google/electra-large-discriminator"
MAX_LEN = 512

label2id = {"elementary": 0, "middle": 1, "high": 2}
id2label = {v: k for k, v in label2id.items()}

CKPT_PATH = f"{DRIVE_OUT_DIR}/{CKPT_NAME}"
print("Checkpoint:", CKPT_PATH)

In [ ]:
# ── Model definition: same architecture used during training ───────────────
import sys
from pathlib import Path

import torch
from transformers import AutoTokenizer

for module_dir in [
    Path.cwd(),
    Path.cwd() / "DomainAdversialNN",
    Path("/content/drive/MyDrive/BeyondFlesch/Beyond-Flesch/DomainAdversialNN"),
    Path("/content/drive/MyDrive/BeyondFK/DomainAdversialNN"),
]:
    if module_dir.exists() and str(module_dir) not in sys.path:
        sys.path.append(str(module_dir))

from dann_models import ElectraScalarMixDANN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Shared DANN model import OK")

In [ ]:
# ── Load checkpoint from Drive ─────────────────────────────────────────────
assert os.path.exists(CKPT_PATH), f"Missing checkpoint: {CKPT_PATH}"

ckpt = torch.load(CKPT_PATH, map_location=device)
domain2id = ckpt.get("domain2id", {"default": 0})
num_domains = len(domain2id)

model = ElectraScalarMixDANN(MODEL_NAME, num_classes=3, num_domains=num_domains).to(device)
model.load_state_dict(ckpt["state_dict"])
model.eval()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loaded:", CKPT_PATH)
print("Checkpoint phase:", ckpt.get("phase", "best_val"))
print("Domains:", domain2id)

In [ ]:
# ── Gradio app ─────────────────────────────────────────────────────────────
import gradio as gr
import torch.nn.functional as F


def classify_reading_level(text: str):
    text = str(text).strip()
    if not text:
        return {"elementary": 0.0, "middle": 0.0, "high": 0.0}

    enc = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LEN,
        padding=True,
        return_tensors="pt",
    )
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        logits = model.difficulty_logits_only(enc["input_ids"], enc["attention_mask"])
        probs = F.softmax(logits, dim=-1).squeeze(0).detach().cpu().numpy()

    return {id2label[i]: float(probs[i]) for i in range(len(probs))}


demo = gr.Interface(
    fn=classify_reading_level,
    inputs=gr.Textbox(
        lines=8,
        placeholder="Paste a sentence, paragraph, or passage here...",
        label="Input text",
    ),
    outputs=gr.Label(num_top_classes=3, label="Predicted education level"),
    title="Beyond-Flesch Reading Level Classifier",
    description=(
        "ELECTRA + ScalarMix classifier trained on Llama-judge labels. "
        "Predicts elementary, middle, or high school reading level."
    ),
    examples=[
        ["The cat sat on the mat. It was happy and warm."],
        ["Climate change affects ecosystems, agriculture, and public health across the world."],
        ["The legislature ratified the constitutional amendment after prolonged bipartisan negotiations."],
    ],
)

demo.launch(share=True)